# S07 — Shorter training window (all four frameworks)

Advisor experiment: cut training to **Jan 2024 onwards for every zone**, keep validation
(2026 Jan-Apr) and Test A (May 2026) unchanged, and see how the four frameworks shift.

Runs `scripts/train_all_frameworks.py` on branch `exp/train-from-2024`:
single-tier prod, single-tier cons, two-tier prod (E2), two-tier cons (E3); 3 seeds,
validation-selected. It **checkpoints after every (zone, framework, seed) and is
resumable** (re-run the *Run* cell after a disconnect). Each cell saves predictions,
per-seed metrics, and the trained model (E2 included via `save_e2`). Outputs are
suffixed `train2024` so the full-window results stay side by side.

## 1. Setup (mount Drive, get the repo on the experiment branch)

In [ ]:
import os, sys, getpass
IN_COLAB = 'google.colab' in sys.modules
REPO = '/content/carbon-intensity-forecast'
DRIVE = '/content/drive/MyDrive/carbon-intensity-forecast'
BRANCH = 'exp/train-from-2024'
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
    if not os.path.isdir(REPO):
        tok = getpass.getpass('GitHub token (repo read): ')
        os.system(f'git clone https://{tok}@github.com/nicolaswilches/carbon-intensity-forecast.git {REPO}')
    os.system(f'cd {REPO} && git fetch -q origin && git checkout -q {BRANCH} && git pull -q')
    DATA_ROOT = f'{DRIVE}/data/processed'
    OUTDIR    = f'{DRIVE}/outputs_train2024'
else:
    REPO = os.path.abspath('..')
    DATA_ROOT = os.path.join(REPO, 'data', 'processed')
    OUTDIR    = os.path.join(REPO, 'outputs')
sys.path.insert(0, os.path.join(REPO, 'src'))
os.environ['KERAS_BACKEND'] = 'tensorflow'
print('repo  :', REPO)
print('branch:', BRANCH)
print('data  :', DATA_ROOT)
print('out   :', OUTDIR)

## 2. Verify the 5 processed parquets are on Drive

In [ ]:
import glob
files = sorted(glob.glob(os.path.join(DATA_ROOT, '*.parquet')))
print(len(files), 'parquet files:', [os.path.basename(f) for f in files])
assert len(files) >= 5, 'Upload data/processed/*.parquet to Drive at ' + DATA_ROOT

## 3. Run (full config, multi-hour on GPU; resumable)

Default does all 5 zones x 4 frameworks x 3 seeds with `TRAIN_START=2024-01`. Because
Colab resets, you can run it in **chunks** and the ledger resumes, e.g. the heavy
two-tier ones per zone:

```python
env = dict(env, FRAMEWORKS='e3_cons', ZONES='BE')   # one heavy cell at a time
```

Re-run this cell after any disconnect.

In [ ]:
env = dict(DATA_ROOT=DATA_ROOT, OUTDIR=OUTDIR, TRAIN_START='2024-01',
           SEEDS='0,1,2', KERAS_BACKEND='tensorflow')
# Optional chunking: env['ZONES']='SG'; env['FRAMEWORKS']='single_cons,e2_prod'
envstr = ' '.join(f'{k}={v}' for k, v in env.items())
!cd {REPO} && {envstr} python scripts/train_all_frameworks.py

## 4. Results: per-zone x framework Test A MAPE (validation-selected)

In [ ]:
import pandas as pd
df = pd.read_csv(os.path.join(OUTDIR, 'all_frameworks_train2024.csv'))
piv = df.pivot(index='zone', columns='framework', values='test_mape')
piv

## 5. Push results back to GitHub

Commit the metrics so they can be pulled locally for the full-window vs 2024-window
comparison. (Models and preds are large and gitignored; keep them on Drive.)

```bash
!cd {REPO} && cp {OUTDIR}/all_frameworks_train2024*.csv outputs/ && \
    git add outputs/all_frameworks_train2024*.csv && \
    git commit -m "exp: 2024-window all-frameworks metrics" && git push origin {BRANCH}
```